# Phase 2 — unchanged Qwen3-4B on DEV (PREREGISTRATION §5 with Amendment 1)
Run this **once, before any training run**. 40 DEV families × E/R/C = 120 generations with the unchanged model.
Frozen protocol (PREREGISTRATION §4): prompt `test1-english-final-v2`, greedy non-thinking decoding, NF4 + FP16, SDPA math, output cap 1,024 tokens, scorer v1.0.0, engine seed 11. **Do not change any setting.**
The notebook embeds `phase2/IMPLEMENTATION_MANIFEST.md` (SHA-256 `414b67eb43db3d7bffd052452634e772e592d1de8e6741e65a73384700db5787`) and refuses to run if any embedded code or data differs from it, or if the pinned tokenizer/config files differ (Amendment 1 item 14). It refuses any stack other than the exact §2 stack (torch 2.11.0+cu130 included) on a T4 (Amendment 2 item 3).
At the end it prints **high unchanged-DEV accuracy (ceiling-risk proxy)** with k/40 and its Wilson 95% interval; the flag fires at C ≥ 38/40 (Amendment 1 item 7). It is a proxy only: Δ_C stays first in the report whatever it shows. DEV is a sanity set: it never selects anything. No FINAL_TEST data is in this notebook.
**Baseline first (Amendment 2 item 6).** The complete run is recorded on Drive (`MyDrive/Hinglish_Phase2/baseline_dev/base_baseline_record.json`: results-ZIP SHA-256 and implementation-manifest SHA-256). The Train notebook refuses the first training attempt without it, and every attempt and the adapter freeze record its ZIP hash. Do not delete or rerun it.
**Stale lock (Amendment 2 item 5).** If a runtime was killed during an evaluation, re-running may stop with "Run lock exists". First make sure no other Colab session is running this notebook, then set `RECOVER_STALE_LOCK = True` and re-run all: the lock is removed, a timestamped record is written next to the run, and generation resumes from the stored journal (nothing is regenerated). Set it back to False afterwards.
**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all** (about 15–20 minutes). Drive is on by default so a disconnect resumes (re-run all; finished prompts are kept, never regenerated). A results ZIP with checksums downloads at the end; send it unchanged.


In [ ]:
import os, sys, json, base64, io, zipfile, hashlib, subprocess, time
from pathlib import Path
USE_DRIVE = True  # resume after a disconnect
RECOVER_STALE_LOCK = False  # Amendment 2 item 5: set True ONLY after confirming that no other Colab session is running this evaluation (a killed runtime left RUNNING.lock); the recovery is logged with a timestamp and stored outputs are never regenerated
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONHASHSEED'] = '0'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'  # download bars would become thousands of streamed lines
PACKAGE = Path('/content/hinglish_phase2') if Path('/content').exists() else Path.cwd() / 'hinglish_phase2'
PACKAGE.mkdir(parents=True, exist_ok=True)
blob = base64.b64decode('...')
assert hashlib.sha256(blob).hexdigest() == 'd3bc3e19a844dfff673a23881bf5235e18eff6ebd12c05c4717d9e4aee78b041'
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(PACKAGE)
INPUTS = PACKAGE / 'inputs'
# Implementation manifest (PREREGISTRATION Amendment 1 item 14): refuse to run if the embedded code differs from it.
MANIFEST_TEXT = base64.b64decode('IyBQaGFzZS0yIGltcGxlbWVudGF0aW9uIG1hbmlmZXN0IChQUkVSRUdJU1RSQVRJT04gQW1lbmRtZW50IDEgaXRlbSAxNCwgQW1lbmRtZW50IDIpCgpHZW5lcmF0ZWQgYnkgYHB5dGhvbjMgLUIgcGhhc2UyL2J1aWxkX2ltcGxlbWVudGF0aW9uX21hbmlmZXN0LnB5YDsgdmVyaWZ5IHdpdGggYC0tY2hlY2tgLiBEbyBub3QgZWRpdCBieSBoYW5kLgoKLSBDb21taXQgdGhpcyBmaWxlICoqYmVmb3JlIHRoZSBmaXJzdCBQaGFzZS0yIERFViBnZW5lcmF0aW9uKiouIFRoZSBERVYtYmFzZWxpbmUgYW5kIFRyYWluIG5vdGVib29rcyBlbWJlZCBpdCB3aXRoIGl0cyBTSEEtMjU2IGFuZCByZWZ1c2UgdG8gcnVuIGlmIGFueSBlbWJlZGRlZCBjb2RlIG9yIGRhdGEgZGlmZmVycyBmcm9tIGl0LgotIFRoZSB0cmFpbmVyIHJlY2hlY2tzIGl0IGFnYWluc3QgdGhlIHJ1bm5pbmcgY29kZSBhbmQgdGhlIGZyb3plbiBUUkFJTi9leHBvc3VyZSBmaWxlcyBiZWZvcmUgZXZlcnkgYXR0ZW1wdCwgYW5kIHJlZnVzZXMgYW4gYXR0ZW1wdCB3aG9zZSBtYW5pZmVzdCBkaWZmZXJzIGZyb20gZWFybGllciBhdHRlbXB0cyBvciBmcm9tIHRoZSB1bmNoYW5nZWQtREVWIGJhc2VsaW5lJ3MuCi0gSXQgaXMgcmVjaGVja2VkIGF0IGFkYXB0ZXIgZnJlZXplIGFnYWluc3QgdGhlIHJlYWwgZmlsZXMgYW5kIGV2ZXJ5IHN0YXJ0IGV2ZW50OiBldmVyeSByZXRhaW5lZCBydW4gcmVjb3JkcyB0aGlzIGZpbGUncyBTSEEtMjU2LCBhbmQgYHBoYXNlMi9BREFQVEVSU19GUk9aRU4ubWRgIGFuZCB0aGUgRmluYWxUZXN0IGdhdGUgcmVxdWlyZSBpdC4KLSBBbnkgY2hhbmdlIHRvIGEgaGFzaGVkIGZpbGUgbmVlZHMgYSByZWdlbmVyYXRlZCBtYW5pZmVzdCBhbmQgcmVidWlsdCBub3RlYm9va3MgYmVmb3JlIHVzZS4KCmBgYGpzb24KewogInNjaGVtYSI6ICJwaGFzZTItaW1wbGVtZW50YXRpb24tbWFuaWZlc3QtMSIsCiAicHJlcmVnaXN0cmF0aW9uIjogewogICJwYXRoIjogInBoYXNlMi9QUkVSRUdJU1RSQVRJT04ubWQiLAogICJzaGEyNTYiOiAiYjJlNmRmOGY1NTJkZmU1NzU2Y2ZkZTkzMDg4ODBmMDZkZTA2ODVkN2QxZDQwZGU1N2M1ZTcxY2M0MTI2MDM5NiIsCiAgImluY2x1ZGVzIjogIkFtZW5kbWVudHMgMSAoRDA0NSkgYW5kIDIgKEQwNDYpIgogfSwKICJncm91cHMiOiB7CiAgInRyYWluZXIiOiBbCiAgICJ0NF9zbW9rZS90cmFpbl9waGFzZTIucHkiLAogICAidDRfc21va2UvdHJhaW5fc21va2UucHkiCiAgXSwKICAibGFiZWxfYnVpbGRlciI6IFsKICAgInQ0X3Ntb2tlL2xhYmVscy5weSIKICBdLAogICJwcm9tcHQiOiBbCiAgICJ0NF9zbW9rZS9wcm90b2NvbC5weSIKICBdLAogICJldmFsdWF0aW9uX2RyaXZlciI6IFsKICAgInQ0X3Ntb2tlL3BoYXNlMl9ldmFsLnB5IgogIF0sCiAgImVuZ2luZSI6IFsKICAgInRlc3QxX3BpbG90L2hpbmdsaXNoX3BpbG90L19faW5pdF9fLnB5IiwKICAgInRlc3QxX3BpbG90L2hpbmdsaXNoX3BpbG90L2RhdGEucHkiLAogICAidGVzdDFfcGlsb3QvaGluZ2xpc2hfcGlsb3QvZW5naW5lLnB5IiwKICAgInRlc3QxX3BpbG90L2hpbmdsaXNoX3BpbG90L2hmX2JhY2tlbmQucHkiLAogICAidGVzdDFfcGlsb3QvaGluZ2xpc2hfcGlsb3QvcmVwb3J0LnB5IiwKICAgInRlc3QxX3BpbG90L2hpbmdsaXNoX3BpbG90L3Njb3Jlcl92MS5weSIKICBdLAogICJzY29yZXJfdjEiOiBbCiAgICJ0ZXN0MV9waWxvdC9oaW5nbGlzaF9waWxvdC9zY29yZXJfdjEucHkiCiAgXSwKICAibnVtZXJpY19saXRlcmFsX3YxIjogWwogICAic2Vjb25kYXJ5X2FuYWx5c2lzL251bWVyaWNfZW5kcG9pbnQucHkiCiAgXSwKICAibnVtZXJpY19saXRlcmFsX3YyIjogWwogICAic2Vjb25kYXJ5X2FuYWx5c2lzL251bWVyaWNfbGl0ZXJhbF92Mi5weSIKICBdLAogICJzZWNvbmRhcnlfcmVwb3J0IjogWwogICAic2Vjb25kYXJ5X2FuYWx5c2lzL19faW5pdF9fLnB5IiwKICAgInNlY29uZGFyeV9hbmFseXNpcy9jbWkucHkiLAogICAic2Vjb25kYXJ5X2FuYWx5c2lzL3JlcG9ydF9zZWNvbmRhcnkucHkiCiAgXSwKICAiYW5hbHlzaXMiOiBbCiAgICJwaGFzZTJfYW5hbHlzaXMvX19pbml0X18ucHkiLAogICAicGhhc2UyX2FuYWx5c2lzL2FuYWx5emUucHkiLAogICAicGhhc2UyX2FuYWx5c2lzL2NhbGlicmF0ZS5weSIsCiAgICJwYXBlcl9hbmFseXNpcy9zYWZlemlwLnB5IgogIF0sCiAgIm5vdGVib29rX2J1aWxkZXJzIjogWwogICAiY29sYWJfbm90ZWJvb2tzL2J1aWxkX3BoYXNlMl9ub3RlYm9va3MucHkiLAogICAicGhhc2UyL2J1aWxkX2ltcGxlbWVudGF0aW9uX21hbmlmZXN0LnB5IgogIF0sCiAgImV4cG9zdXJlX29yZGVyIjogWwogICAicGhhc2UyL0VYUE9TVVJFX09SREVSLmpzb24iCiAgXSwKICAiZnJvemVuX2RhdGEiOiBbCiAgICJwaGFzZTIvdHJhaW5fQS52Mi5qc29uIiwKICAgInBoYXNlMi90cmFpbl9CLnYyLmpzb24iLAogICAicGhhc2UyL2Rldi52Mi5qc29uIgogIF0KIH0sCiAiZmlsZXMiOiB7CiAgImNvbGFiX25vdGVib29rcy9idWlsZF9waGFzZTJfbm90ZWJvb2tzLnB5IjogImUxZDkzMmEyOTM3OGQwMTE3Y2MwZTBhN2Q2OGRkMzJmNTczZDU5MDU4MWI3MDcyMGVmNzViYTU2ZDk0MGFkZWUiLAogICJwYXBlcl9hbmFseXNpcy9zYWZlemlwLnB5IjogImYwOTdhOWI4MWE2OGYzNTY0YzI3YWU2MGQzOGE1M2M4MzRjZWU1YzM4YjRhMjBiNzM5ZGZhZTBjM2RkNjU1M2QiLAogICJwaGFzZTIvRVhQT1NVUkVfT1JERVIuanNvbiI6ICJhZGY0M2U1MGM4MGM1OWZmYTgzMTA3MTU1ODc4NTQ2OWIxZWRhODk4ODBjMThlZWZiNGY4OWE5YWRlYjVhZmI2IiwKICAicGhhc2UyL2J1aWxkX2ltcGxlbWVudGF0aW9uX21hbmlmZXN0LnB5IjogImE5N2FkZDVkNzE3YTRjOGM4ZjY4YWY3ZTA3N2ZlODRjZWFlZmU5YWY2NzUxOGE2ZjljMTQxZWEzYWIwNTUzMjUiLAogICJwaGFzZTIvZGV2LnYyLmpzb24iOiAiZTc0MWMxNjUyZjViM2RjYWViYzc4ZDlkZDlkN2MzMWZkZjQ0MTczMDZmYmIyZjA4NjIxNDlmMjdmODU4ZjI4YiIsCiAgInBoYXNlMi90cmFpbl9BLnYyLmpzb24iOiAiYjk2NTZhOGM0MWFmMmE3OTViY2U5YzEzMWVmYzZhNWVmYzY1MmRmNzgzNjhkMTRmYjc3MjBmZDk4NTg4MzY2NiIsCiAgInBoYXNlMi90cmFpbl9CLnYyLmpzb24iOiAiYzk3NWZlMTI0NTlkZjYxNjBlZmZjNWFlYjAwODdmOTQ2NjZiYTBiNDk3NzhjM2Q4MjdhNjdiYjkyM2ZmMzViZCIsCiAgInBoYXNlMl9hbmFseXNpcy9fX2luaXRfXy5weSI6ICJiNzJkZjQxNWExY2M4NGUzMDdiODFmZWUwYjEzZTZlM2MxZTdjMDQ0NDhhOTU1Y2ZmZGY1Y2M4MmMyYjc3ODgzIiwKICAicGhhc2UyX2FuYWx5c2lzL2FuYWx5emUucHkiOiAiMWU5YmI2OTlkOGVlMzRmMWZhODcxZTRjOTM2ZTE3YTZhYzlkOGIyMTFiY2IwYmMwZTM5ODliOTA0NjVjNWZkMiIsCiAgInBoYXNlMl9hbmFseXNpcy9jYWxpYnJhdGUucHkiOiAiOTNjZGZjMzYwZjAzZTRiODI4ZjE0MTc3NzczZGI1ZmQ0MzQyNDIxYTQ3OWU3MmQ4Nzg3YmIyZTdlOGE3ZDUyZCIsCiAgInNlY29uZGFyeV9hbmFseXNpcy9fX2luaXRfXy5weSI6ICJmMDY2N2ZiMzI4NDcyODA2MzA4NTYwYjNhMjM3MTA3ZTVkNWNkYTdlZTdhZjAxYjM1ZmVhZjhlZGMwZmUzYzI1IiwKICAic2Vjb25kYXJ5X2FuYWx5c2lzL2NtaS5weSI6ICI2NTQ5YTczZGIwODc1Yjk1NDYzZTE1ZTQ2ODU2MmM4ZDk2MzY5YTNkYzYxMmRiODNhNDBkNDU4ZjBlNzc5NDg5IiwKICAic2Vjb25kYXJ5X2FuYWx5c2lzL251bWVyaWNfZW5kcG9pbnQucHkiOiAiY2U3MGFlZmQxZjRjMDNmZmU4MGJiZDA2ZDRkZjAyNWFiZTkyZmMwODY1NzMyYjZkMmU2ZGIxMzUwNGVkODgzZiIsCiAgInNlY29uZGFyeV9hbmFseXNpcy9udW1lcmljX2xpdGVyYWxfdjIucHkiOiAiODAyNjlhYzYyZDA5Y2UyYmRlZTMyNzFlOWVkMjUzYWQ0YTUyZWM1MWRlMTA1YzNlZDdkYzkxOWQ4MWE3MTJiNCIsCiAgInNlY29uZGFyeV9hbmFseXNpcy9yZXBvcnRfc2Vjb25kYXJ5LnB5IjogImJmYTFhZGJhMmJlZmRkNzc0NTkyYjU3Mzk2NmI1YzA2MzE5YzMyNTRjMGNkYzVhOWMwZWYyODRhMDEwN2JhMGMiLAogICJ0NF9zbW9rZS9sYWJlbHMucHkiOiAiMjM2YmIzYTcwZjVlNzRkZWIyY2MwNjc3MDgyM2Q5MDljMGRmMThkYjJlOWJiOWYzZWFmMDVmNTFiZGE2OWEzNyIsCiAgInQ0X3Ntb2tlL3BoYXNlMl9ldmFsLnB5IjogIjU5NjVhYWE3YjkxNWMzNmM2ZjMyMzEyZDE1MWY5N2JmNTk0Nzk5ZTdiMjVhYzBiMjg5NTIxZDZhNmIyMWFlOGIiLAogICJ0NF9zbW9rZS9wcm90b2NvbC5weSI6ICJlNjgzODQxNmE1YTZjMzk5ZjE5ZWJkZDBhNzYwYzdmZDBhMTAxNDBkZDZkZTA0YWEzMDkyNjA5M2NjNTM3ZjE2IiwKICAidDRfc21va2UvdHJhaW5fcGhhc2UyLnB5IjogIjE0MDlhNWUyNmNhNzZmMDkxNjA4ZTY4ZjAxNjc4M2ZjMWYxOTZlOTgzMmIzMmY2Mjc5ZTEwZjUzMjVkZTQxYTYiLAogICJ0NF9zbW9rZS90cmFpbl9zbW9rZS5weSI6ICIwNWZhOTRmZGRkYTEyYjlmOTljMDUwY2IzZWIwMjU2Mjg3NWVhMGYzZmQzMTIwNDE5MTM2NDM2N2RkNDE1ZjQ3IiwKICAidGVzdDFfcGlsb3QvaGluZ2xpc2hfcGlsb3QvX19pbml0X18ucHkiOiAiZjgzMDYwMDJmNWJkYTFkMjc1MzVlMWU2YTRjNDI1NWUyMmIwNDU0ODlmYjgwN2IwYTc0NjBlNWUzMDgwMzZjZiIsCiAgInRlc3QxX3BpbG90L2hpbmdsaXNoX3BpbG90L2RhdGEucHkiOiAiNTE5Zjg5OTVhNjBjZDE5MzM3NWVhZWNlZTJmODk4MDFhMmVlYThkMDY0NzQyMDllYWI0MDU3NDk4ODZhMTYyYyIsCiAgInRlc3QxX3BpbG90L2hpbmdsaXNoX3BpbG90L2VuZ2luZS5weSI6ICI4MDdlNjEwMWQwODRhZjU1YWI4MTM0YTI0MTQ5MTNkM2NlMDM2M2ZkODk3MmU2YTMxMmUzNWNjOTA4NmJiODhiIiwKICAidGVzdDFfcGlsb3QvaGluZ2xpc2hfcGlsb3QvaGZfYmFja2VuZC5weSI6ICIyZTlmYjRiMGMxZjY4ZjMwZThmMGZjODRmMTkyNzE2NWVhYjZiZDk2OTRiOTQ1MDc5ZDYwM2U1N2ViYTc4YWZjIiwKICAidGVzdDFfcGlsb3QvaGluZ2xpc2hfcGlsb3QvcmVwb3J0LnB5IjogIjAyZGRjNTE1NTBlYWY1OGNiYzQ1MTUwZDhkZTM4MzM3NGM2ZmI5YzRhY2U2ZDkxNWExMzNlZWNlZDk0ZTY4YzQiLAogICJ0ZXN0MV9waWxvdC9oaW5nbGlzaF9waWxvdC9zY29yZXJfdjEucHkiOiAiYzQ4ZjlkMTJlNzMzZWIxYmZlNWFmNWIxZmVjZTZlM2ZmNjljYjIzNDJhOTdmZTNiYThhYzlmMDFmNjhkYmJiYyIKIH0sCiAicHJlcmVnaXN0ZXJlZF9oYXNoZXNfdmVyaWZpZWQiOiB7CiAgInRlc3QxX3BpbG90L2hpbmdsaXNoX3BpbG90L3Njb3Jlcl92MS5weSI6ICJjNDhmOWQxMmU3MzNlYjFiZmU1YWY1YjFmZWNlNmUzZmY2OWNiMjM0MmE5N2ZlM2JhOGFjOWYwMWY2OGRiYmJjIiwKICAic2Vjb25kYXJ5X2FuYWx5c2lzL251bWVyaWNfZW5kcG9pbnQucHkiOiAiY2U3MGFlZmQxZjRjMDNmZmU4MGJiZDA2ZDRkZjAyNWFiZTkyZmMwODY1NzMyYjZkMmU2ZGIxMzUwNGVkODgzZiIKIH0sCiAibm90ZWJvb2tfcGF5bG9hZHMiOiB7CiAgIlBoYXNlMl9CYXNlbGluZV9ERVYuaXB5bmIiOiAiZDNiYzNlMTlhODQ0ZGZmZjY3M2EyMzg4MWJmNTIzNWUxOGVmZjZlYmQxMmMwNWM0NzE3ZDllNGFlZTc4YjA0MSIsCiAgIlBoYXNlMl9UcmFpbi5pcHluYiI6ICJjYzM5ODYzYTM0Yzk2NGFjZGRhNWNhYjEwY2IxOWU1OTk5YWU2OGQxNDJhMzRkYmJjMmQzZDNiZDRkYzE3NWMyIgogfSwKICJub3RlYm9va19wYXlsb2FkX25vdGUiOiAiU0hBLTI1NiBvZiB0aGUgY29kZStkYXRhIFpJUCBlbWJlZGRlZCBpbiBlYWNoIG5vdGVib29rOyB0aGUgY2VsbHMgYXJlIHJlbmRlcmVkIGJ5IGNvbGFiX25vdGVib29rcy9idWlsZF9waGFzZTJfbm90ZWJvb2tzLnB5IChoYXNoZWQgYWJvdmUpLiBQaGFzZTJfRmluYWxUZXN0LmlweW5iIGlzIGJ1aWx0IG9ubHkgYWZ0ZXIgQURBUFRFUlNfRlJPWkVOLm1kLCBmcm9tIHRoZSBzYW1lIGJ1aWxkZXIsIGFuZCB2ZXJpZmllcyB0aGlzIG1hbmlmZXN0LiIsCiAiZXhwb3N1cmVfb3JkZXIiOiB7CiAgInBhdGgiOiAicGhhc2UyL0VYUE9TVVJFX09SREVSLmpzb24iLAogICJzaGEyNTYiOiAiYWRmNDNlNTBjODBjNTlmZmE4MzEwNzE1NTg3ODU0NjliMWVkYTg5ODgwYzE4ZWVmYjRmODlhOWFkZWI1YWZiNiIsCiAgIm9yZGVyX3NoYTI1NiI6IHsKICAgIjExIjogImJlOWFhOTBiNzYzMjczZmQyMzlhOTNkZGVmYjAzNWNhOWM5ZTIyMDE3NDk5NTY1MmUyNDZmNzg4Nzc3YzNiYTciLAogICAiMjIiOiAiZjljYzBlYjdiZjdkMjQ5MWJjYjQ0YjRkYTEwNDUyZjM4MDQ5ZjQ4YjM1ZTIyMmJiZWVhNTkzNDVlZDU1MjM1ZSIsCiAgICIzMyI6ICI5ZWNmMmRjMTQ5MjY0ODYzMjMzZThhOGQ4Zjc5Yjk4ZmIxMTc4ZTZkZDcyNWVjMmQ4NDZkZGViYTlhYTkwYTAzIiwKICAgIjQ0IjogIjNjNmE2NjQ3YWRmNTZhOWViYTM4M2IyZGMzZWM3ZGI3ZWMwZWQ4YWRmYTRkNzY5YjYzZWZjMzc3YWRkMGFjN2MiLAogICAiNTUiOiAiZDRiYzhjODA5ZGRjODA1NjU2NzllZWEwZjkzNTM2MTJlY2M2YWEyM2JkM2RjM2MzODcxMTdjMDU4NGVkMDBkOSIKICB9CiB9LAogInByb21wdCI6IHsKICAidmVyc2lvbiI6ICJ0ZXN0MS1lbmdsaXNoLWZpbmFsLXYyIiwKICAiaW5zdHJ1Y3Rpb25fc2hhMjU2IjogIjNkMGI1MTZjOWZiNTk3ODc1NWUxMmJkZDJhOGIzYzE3NDJiNjA1ZTJjM2VkYzJlMzlmYzJjNTMxMTVkYzdkNjMiLAogICJpZGVudGljYWxfaW4iOiBbCiAgICJ0NF9zbW9rZS9wcm90b2NvbC5weSIsCiAgICJ0ZXN0MV9waWxvdC9oaW5nbGlzaF9waWxvdC9lbmdpbmUucHkiCiAgXQogfSwKICJtb2RlbCI6IHsKICAiaWQiOiAiUXdlbi9Rd2VuMy00QiIsCiAgInJldmlzaW9uIjogIjFjZmE5YTcyMDg5MTIxMjY0NTkyMTRlOGIwNDMyMTYwM2IzZGY2MGMiLAogICJ0b2tlbml6ZXJfcmV2aXNpb24iOiAiMWNmYTlhNzIwODkxMjEyNjQ1OTIxNGU4YjA0MzIxNjAzYjNkZjYwYyIsCiAgImh1Yl9maWxlcyI6IHsKICAgImNvbmZpZy5qc29uIjogIjhiYTAwNmY3NGZlY2ZhYWViMzkyODcyYTYwZjRhNDgwZTdlYzk4NjAxNTNkMmUxYjc2OWVjODFmOWExNDdmOGEiLAogICAiZ2VuZXJhdGlvbl9jb25maWcuanNvbiI6ICIyMzI1ZGEwZjE1YmI4NDhlMDE4YzVhZTA3MWI3OTQzMzMyZTlmODcxZDZiNjBlMmVkMjJjYTk3ZDRjYjk5M2QyIiwKICAgIm1lcmdlcy50eHQiOiAiODgzMWU0ZjFhMDQ0NDcxMzQwZjdjMGE4M2Q3YmQ3MTMwNmE1Yjg2N2U5NWZkODcwZjc0ZDBjNTMwOGE5MDRkNSIsCiAgICJtb2RlbC5zYWZldGVuc29ycy5pbmRleC5qc29uIjogIjZkYzA5ODFiODgyOWZlYWQ3NDY0NDFmNjhmMzhmMjRjNWNhNGEzYTY2MzUxZjY1MmMyNmM2ZGYwZWZjNDNhYjIiLAogICAidG9rZW5pemVyLmpzb24iOiAiYWViMTMzMDdhNzFhY2Q4ZmU4MTg2MWQ5NGFkNTRhYjY4OWRmNzczMzE4ODA5ZWVkM2NiZTc5NGI0NDkyZGFlNCIsCiAgICJ0b2tlbml6ZXJfY29uZmlnLmpzb24iOiAiZDVkMDlmMDdiNDhjMzA4NmM1MDhiMzBkMWM5MTE0YmQxMTg5MTQ1Yjc0ZTk4MmEyNjUzNTBjOTIzYWNkODEwMSIsCiAgICJ2b2NhYi5qc29uIjogImNhMTBkN2U5ZmIzZWQxODU3NWRkMWUyNzdhMjU3OWMxNmQxMDhlMzJmMjc0Mzk2ODRhZmEwZTEwYjE0NDA5MTAiCiAgfSwKICAiY2hhdF90ZW1wbGF0ZSI6IHsKICAgInNvdXJjZSI6ICJ0b2tlbml6ZXJfY29uZmlnLmpzb24gYXQgdGhlIHBpbm5lZCByZXZpc2lvbiAobm8gY2hhdF90ZW1wbGF0ZS5qaW5qYSBleGlzdHMgdGhlcmUpIiwKICAgImRpZ2VzdCI6ICI0MWQ1OTI5YmY3Mzc5NmJlYjY2ODA5YWM3MDBiMmNmM2ZmODE2OTRmOTMzZTVjMTRkNTJiN2ZkNjk2M2M5NDdkIiwKICAgImRpZ2VzdF9kZWZpbml0aW9uIjogInRyYWluX3Ntb2tlLmRpZ2VzdCh0ZW1wbGF0ZSkgPSBTSEEtMjU2IG9mIGpzb24uZHVtcHModGVtcGxhdGUpIGluIFVURi04OyByZWNvcmRlZCBhdCBydW4gdGltZSBhcyBjaGF0X3RlbXBsYXRlX3NoYTI1NiBieSB0aGUgdHJhaW5lciBhbmQgYnkgdGhlIGVuZ2luZSBiYWNrZW5kIiwKICAgInV0Zjhfc2hhMjU2IjogImE1NWVlMWIxNjYwMTI4YjcwOTg3MjNlMGFiY2Q5MmNhYTA3ODgwNjEwNTFjNjJkNTFjYmU4N2Q5Y2YxOTc0ZDgiLAogICAia3dhcmdzIjogewogICAgImVuYWJsZV90aGlua2luZyI6IGZhbHNlLAogICAgImFkZF9nZW5lcmF0aW9uX3Byb21wdCI6IHRydWUKICAgfQogIH0sCiAgInRydXN0X3JlbW90ZV9jb2RlIjogZmFsc2UsCiAgInNlY29uZF9iYWNrYm9uZSI6ICJub25lIChBbWVuZG1lbnQgMSBpdGVtIDE1OiBMbGFtYS0zLjItM0IgZXhjbHVkZWQpIgogfSwKICJnZW5lcmF0aW9uX2NvbmZpZyI6IHsKICAiZW5naW5lX2NvbmZpZ19zaGEyNTYiOiB7CiAgICJkZXYiOiAiZTgyODA2MjJmOWNmNDVmYmI2MmJiNTM3YzE4ODNjNWIzOTRhN2E4ZDIzZTg1MTY0ZGJiN2IyOTdlZWM4ZjViNiIsCiAgICJmaW5hbF90ZXN0IjogImQ3MjlhZGE0OTdjZGQ2MTRmMjU4NjU4YTM5NzNhMmI3OTMwNWY0ZTE4MjE5MjIwMjA4YmU1N2RkMzQ5MmJjYTgiCiAgfSwKICAiZGVjb2RpbmciOiB7CiAgICJtb2RlbF9pZCI6ICJRd2VuL1F3ZW4zLTRCIiwKICAgInNlZWQiOiAxMSwKICAgIm1heF9uZXdfdG9rZW5zIjogMTAyNCwKICAgImNvbnRleHRfbGltaXQiOiAyMDQ4LAogICAidGhpbmtpbmciOiBmYWxzZSwKICAgImRvX3NhbXBsZSI6IGZhbHNlLAogICAibnVtX2JlYW1zIjogMSwKICAgInF1YW50aXphdGlvbiI6ICJuZjQiLAogICAiZG91YmxlX3F1YW50aXphdGlvbiI6IHRydWUsCiAgICJjb21wdXRlX2R0eXBlIjogImZsb2F0MTYiLAogICAiYXR0ZW50aW9uX2JhY2tlbmQiOiAic2RwYV9tYXRoIiwKICAgIm1heF9pbmZyYXN0cnVjdHVyZV9yZXRyaWVzIjogMSwKICAgInByb21wdF92ZXJzaW9uIjogInRlc3QxLWVuZ2xpc2gtZmluYWwtdjIiCiAgfSwKICAiaGZfZ2VuZXJhdGlvbl9jb25maWciOiB7CiAgICJtYXhfbmV3X3Rva2VucyI6IDEwMjQsCiAgICJkb19zYW1wbGUiOiBmYWxzZSwKICAgIm51bV9iZWFtcyI6IDEsCiAgICJ1c2VfY2FjaGUiOiB0cnVlLAogICAiZW9zX3Rva2VuX2lkIjogIm1vZGVsLmdlbmVyYXRpb25fY29uZmlnLmVvc190b2tlbl9pZCIsCiAgICJwYWRfdG9rZW5faWQiOiAxNTE2NDMsCiAgICJyZXBldGl0aW9uX3BlbmFsdHkiOiAxLjAsCiAgICJ0ZW1wZXJhdHVyZSI6IDEuMCwKICAgInRvcF9wIjogMS4wLAogICAidG9wX2siOiA1MAogIH0sCiAgImVuZ2luZV9zZWVkIjogMTEsCiAgInJldHJ5X3BvbGljeSI6ICJhdCBtb3N0IG9uZSB1bmNoYW5nZWQtY29uZmlndXJhdGlvbiBpbmZyYXN0cnVjdHVyZSByZXRyeSBwZXIgcHJvbXB0OyBhIHN1Y2Nlc3NmdWxseSBnZW5lcmF0ZWQgcmVzcG9uc2UgaXMgbmV2ZXIgcmV0cmllZDsgT09NIG5ldmVyIHJldHJpZWQiLAogICJmaW5hbF90ZXN0X291dHB1dCI6ICJzZWFsZWQ6IGNvbXBsZXRpb24sIGNoZWNrc3VtIGFuZCBpbmZyYXN0cnVjdHVyZSBzdGF0dXMgb25seSAoQW1lbmRtZW50IDEgaXRlbSAxNCkiCiB9LAogImVvc19zdG9wcGluZyI6IHsKICAidHJhaW5pbmdfdGFyZ2V0IjogInJlZmVyZW5jZV9zb2x1dGlvbl9lbiArICc8fGltX2VuZHw+JyAoaWQgMTUxNjQ1KSwgZW5jb2RlZCBzZXBhcmF0ZWx5IGZyb20gdGhlIHByb21wdDsgdGhlIGxhc3Qgc3VwZXJ2aXNlZCB0b2tlbiBpcyBhc3NlcnRlZCB0byBiZSA8fGltX2VuZHw+IiwKICAiZ2VuZXJhdGlvbl9lb3NfdG9rZW5faWQiOiBbCiAgIDE1MTY0NSwKICAgMTUxNjQzCiAgXSwKICAiZ2VuZXJhdGlvbl9lb3Nfc291cmNlIjogIm1vZGVsLmdlbmVyYXRpb25fY29uZmlnLmVvc190b2tlbl9pZCBmcm9tIGdlbmVyYXRpb25fY29uZmlnLmpzb24gYXQgdGhlIHBpbm5lZCByZXZpc2lvbiAoPHxpbV9lbmR8PiAxNTE2NDUsIDx8ZW5kb2Z0ZXh0fD4gMTUxNjQzKTsgcGFkX3Rva2VuX2lkID0gdG9rZW5pemVyIHBhZCA8fGVuZG9mdGV4dHw+IDE1MTY0MyIsCiAgInN0b3BwaW5nIjogImdyZWVkeSBkZWNvZGluZyBzdG9wcyBhdCB0aGUgZmlyc3QgZ2VuZXJhdGVkIEVPUyBpZCBvciBhZnRlciBtYXhfbmV3X3Rva2VucyA9IDEwMjQgbmV3IHRva2VucyIsCiAgInRydW5jYXRlZCI6ICJubyBFT1MgYW1vbmcgdGhlIGdlbmVyYXRlZCBpZHMgYW5kIDEwMjQgdG9rZW5zIGdlbmVyYXRlZDsgc2NvcmVkIDAgYnkgc2NvcmVyIHYxIiwKICAiZGVjb2Rpbmdfb2ZfcmVzcG9uc2UiOiAidG9rZW5pemVyLmRlY29kZShpZHMsIHNraXBfc3BlY2lhbF90b2tlbnM9VHJ1ZSk7IHRoZSByYXcgZm9ybSB3aXRoIHNwZWNpYWwgdG9rZW5zIGlzIGFsc28gc3RvcmVkIiwKICAicHJvbXB0IjogImNoYXQgdGVtcGxhdGUgd2l0aCBzeXN0ZW0gPSBmcm96ZW4gaW5zdHJ1Y3Rpb24sIHVzZXIgPSByZW5kZXJpbmcsIGFkZF9nZW5lcmF0aW9uX3Byb21wdD1UcnVlLCBlbmFibGVfdGhpbmtpbmc9RmFsc2UgKGVtcHR5IDx0aGluaz48L3RoaW5rPiBibG9jayk7IGlkZW50aWNhbCBhdCB0cmFpbmluZyBhbmQgZXZhbHVhdGlvbiIKIH0sCiAiZHR5cGVfcHJlcGFyYXRpb24iOiB7CiAgInRyYWluIjogWwogICAiQXV0b01vZGVsRm9yQ2F1c2FsTE0uZnJvbV9wcmV0cmFpbmVkKHRvcmNoX2R0eXBlPWZsb2F0MTYsIEJpdHNBbmRCeXRlc0NvbmZpZyBuZjQsIGRvdWJsZSBxdWFudCwgY29tcHV0ZSBmbG9hdDE2KSwgc2RwYSBhdHRlbnRpb24sIGRldmljZV9tYXAgeycnOiAwfSIsCiAgICJwZWZ0LnByZXBhcmVfbW9kZWxfZm9yX2tiaXRfdHJhaW5pbmcodXNlX2dyYWRpZW50X2NoZWNrcG9pbnRpbmc9VHJ1ZSwgdXNlX3JlZW50cmFudD1GYWxzZSk6IGNhc3RzIHRoZSBub24tcXVhbnRpemVkIHBhcmFtZXRlcnMgKG5vcm1zLCB0aWVkIGVtYmVkZGluZy9sbV9oZWFkKSB0byBmbG9hdDMyIiwKICAgIkxvUkEgcjE2L2FscGhhMzIvZHJvcG91dCAwLjA1IG9uIHEvay92L28vZ2F0ZS91cC9kb3duOyB0cmFpbmFibGUgcGFyYW1ldGVycyBmb3JjZWQgdG8gZmxvYXQzMiIsCiAgICJmb3J3YXJkIHVuZGVyIHRvcmNoLmF1dG9jYXN0KCdjdWRhJywgZmxvYXQxNik7IGxvc3MgaW4gZmxvYXQzMjsgQWRhbVcgc3RhdGUgZmxvYXQzMjsgR3JhZFNjYWxlciBpbml0IDY1NTM2LCBncm93dGggMi4wLCBiYWNrb2ZmIDAuNSwgaW50ZXJ2YWwgMjAwMDsgU0RQQSBtYXRoIGtlcm5lbCBvbmx5OyB1c2VfY2FjaGUgRmFsc2UiCiAgXSwKICAiZXZhbHVhdGlvbiI6IFsKICAgIkF1dG9Nb2RlbEZvckNhdXNhbExNLmZyb21fcHJldHJhaW5lZCh0b3JjaF9kdHlwZT1mbG9hdDE2LCBCaXRzQW5kQnl0ZXNDb25maWcgbmY0LCBkb3VibGUgcXVhbnQsIGNvbXB1dGUgZmxvYXQxNiksIHNkcGEgYXR0ZW50aW9uLCBtb2RlbC5ldmFsKCksIG5vIHByZXBhcmVfbW9kZWxfZm9yX2tiaXRfdHJhaW5pbmcgKG5vcm1zIGFuZCBlbWJlZGRpbmdzIHN0YXkgZmxvYXQxNikiLAogICAiUGVmdE1vZGVsLmZyb21fcHJldHJhaW5lZChiYXNlLCBhZGFwdGVyLCBpc190cmFpbmFibGU9RmFsc2UpOiB0aGUgZmxvYXQzMiBMb1JBIHdlaWdodHMgYXJlIGxvYWRlZCB1bm1lcmdlZCAobm8gbWVyZ2UsIG5vIHJlLXF1YW50aXphdGlvbik7IG5vIGF1dG9jYXN0IiwKICAgImluZmVyZW5jZV9tb2RlLCBTRFBBIG1hdGgga2VybmVsIG9ubHksIHVzZV9jYWNoZSBUcnVlLCBiYXRjaCAxLCBkZXRlcm1pbmlzdGljIGFsZ29yaXRobXMiCiAgXSwKICAiZGlmZmVyZW5jZSI6ICJ0cmFpbmluZyB1cGNhc3RzIG5vcm1zL2VtYmVkZGluZ3MgdG8gZmxvYXQzMiBhbmQgcnVucyB1bmRlciBmbG9hdDE2IGF1dG9jYXN0OyBldmFsdWF0aW9uIGtlZXBzIHRoZSBmcm96ZW4gZW5naW5lJ3MgZmxvYXQxNiBiYXNlLiBTdGFuZGFyZCBRTG9SQSBwcmFjdGljZSwgZml4ZWQgYmVmb3JlIGFueSBQaGFzZS0yIG91dHB1dC4iCiB9LAogInRyYWluaW5nIjogewogICJyZWNpcGVfc2hhMjU2IjogImI3Mzg0OWZlNjQ5MjYyYTdhZmVlMDVlZWJiOTdlZjdhNjM1NTE0MWRjZDU0ZjM4NWIwMTAxOTVkMjk2ODc5YjQiLAogICJncmFkX3NjYWxlciI6IHsKICAgImluaXRfc2NhbGUiOiA2NTUzNi4wLAogICAiZ3Jvd3RoX2ZhY3RvciI6IDIuMCwKICAgImJhY2tvZmZfZmFjdG9yIjogMC41LAogICAiZ3Jvd3RoX2ludGVydmFsIjogMjAwMAogIH0sCiAgInZhbGlkX3J1biI6ICJleGFjdGx5IDYwIGFjY3VtdWxhdGlvbiB3aW5kb3dzIGFuZCA2MCBzdWNjZXNzZnVsIG9wdGltaXplciBzdGVwcyBvdmVyIGV4YWN0bHkgMyBwYXNzZXM7IGxyKHUpIGlzIHNldCBiZWZvcmUgc3RlcCB1OyBncmFkaWVudHMgYXJlIHVuc2NhbGVkIGJlZm9yZSBjbGlwcGluZzsgYW55IEdyYWRTY2FsZXIgc2tpcCwgbm9uLWZpbml0ZSBsb3NzIG9yIGdyYWRpZW50LCBvciBub24tZmluaXRlIHNhdmVkIHBhcmFtZXRlciBpcyBhIG51bWVyaWNhbCBmYWlsdXJlIChBbWVuZG1lbnQgMSBpdGVtIDEwKSIsCiAgImZhaWx1cmVfcG9saWN5IjogIkFtZW5kbWVudCAxIGl0ZW0gMTE6IGluZnJhc3RydWN0dXJlIC0+IG9uZSBmcm9tLXNjcmF0Y2ggcmV0cnk7IG51bWVyaWNhbC9wcm90b2NvbCAtPiBzdHVkeSBoYWx0cyIsCiAgInRyYWluX2ZhbWlsaWVzX3NoYTI1NiI6ICJkYjFiZWUyY2UyYjQxYzEyYWU4ZWQ2MDAzNTVhMjRkNmRjODk4NWVlMTE4MWMxZTg2ZTIyZmZjNzIzYTQ2N2ZjIiwKICAidHJhaW5lcl9jb2RlX3ZlcmlmaWVkX2F0X2VudHJ5IjogWwogICAidDRfc21va2UvdHJhaW5fcGhhc2UyLnB5IiwKICAgInQ0X3Ntb2tlL3RyYWluX3Ntb2tlLnB5IiwKICAgInQ0X3Ntb2tlL2xhYmVscy5weSIsCiAgICJ0NF9zbW9rZS9wcm90b2NvbC5weSIKICBdLAogICJmYWlsdXJlX2NsYXNzZXMiOiB7CiAgICJpbmZyYXN0cnVjdHVyZSI6IFsKICAgICJydW50aW1lIGRpc2Nvbm5lY3Qgb3Iga2lsbCIsCiAgICAiQ1VEQSBvdXQtb2YtbWVtb3J5IiwKICAgICJEcml2ZSBvciBmaWxlc3lzdGVtIE9TRXJyb3IiLAogICAgIm5ldHdvcmsgb3IgSHViIGRvd25sb2FkIGVycm9yIiwKICAgICJmYWlsZWQgY2hlY2twb2ludCB3cml0ZSIKICAgXSwKICAgInByb3RvY29sIjogImV2ZXJ5IFByb3RvY29sRXJyb3IgKGhhc2gsIGlkZW50aXR5LCBsYWJlbCBvciBkYXRhIGFzc2VydGlvbiwgd2hlcmV2ZXIgZGV0ZWN0ZWQsIGluY2x1ZGluZyBhdCByZWxvYWQpIGFuZCBldmVyeSBvdGhlciBleGNlcHRpb24sIGUuZy4gVHlwZUVycm9yIiwKICAgIm51bWVyaWNhbCI6ICJHcmFkU2NhbGVyIHNraXAsIG5vbi1maW5pdGUgbG9zcywgZ3JhZGllbnQgb3Igc2F2ZWQgcGFyYW1ldGVyIiwKICAgInJ1bGUiOiAiQW1lbmRtZW50IDIgaXRlbSAyOiB1bmtub3duIGVycm9ycyBuZXZlciBlYXJuIGEgcmV0cnkgb3IgYSByZXNlcnZlIgogIH0sCiAgInJ1bl9vcmRlciI6IHsKICAgInNsb3RzIjogWwogICAgMTEsCiAgICAyMiwKICAgIDMzCiAgIF0sCiAgICJhcm1zIjogWwogICAgIkVDIiwKICAgICJFUiIsCiAgICAiRUUiCiAgIF0sCiAgICJyZXNlcnZlX3F1ZXVlIjogWwogICAgNDQsCiAgICA1NQogICBdLAogICAicnVsZSI6ICJBbWVuZG1lbnQgMiBpdGVtIDQ6IHRoZSBhdHRlbXB0IHBsYW4gY2hvb3NlcyB0aGUgbmV4dCBwZW5kaW5nIHJ1bjsgYSBzbG90IGlzIGZ1bGx5IHNldHRsZWQgYmVmb3JlIHRoZSBuZXh0IHNsb3Qgc3RhcnRzIgogIH0sCiAgImJhc2VsaW5lX2ZpcnN0IjogewogICAicmVjb3JkIjogImJhc2VfYmFzZWxpbmVfcmVjb3JkLmpzb24iLAogICAicHJvbXB0cyI6IDEyMCwKICAgInJ1bGUiOiAiQW1lbmRtZW50IDIgaXRlbSA2OiBubyB0cmFpbmluZyBhdHRlbXB0IHdpdGhvdXQgYSBjb21wbGV0ZSB1bmNoYW5nZWQtbW9kZWwgREVWIHJ1biB1bmRlciB0aGlzIG1hbmlmZXN0OyBpdHMgcmVzdWx0cy1aSVAgU0hBLTI1NiBpcyBpbiBldmVyeSBzdGFydCBldmVudCBhbmQgaW4gQURBUFRFUlNfRlJPWkVOLm1kIgogIH0sCiAgImlkZW50aWNhbF9hY3Jvc3NfYXR0ZW1wdHMiOiAiQW1lbmRtZW50IDIgaXRlbSA3OiBjb25maWd1cmF0aW9uIGFuZCBpbXBsZW1lbnRhdGlvbiBpZGVudGl0eSBpbiBldmVyeSBzdGFydCBldmVudDsgYSBkaWZmZXJlbnQgb25lIGlzIHJlZnVzZWQiCiB9LAogImRlcGVuZGVuY2llcyI6IHsKICAicHl0aG9uIjogInJlY29yZGVkIHBlciBydW4gKENvbGFiIGRlZmF1bHQpIiwKICAidG9yY2giOiAiMi4xMS4wK2N1MTMwIChDb2xhYiBidWlsZDsgbmV2ZXIgaW5zdGFsbGVkOyByZXF1aXJlZCBleGFjdGx5KSIsCiAgInRvcmNoX2J1aWxkIjogIjIuMTEuMCtjdTEzMCIsCiAgInN0YWNrX3BvbGljeSI6ICJBbWVuZG1lbnQgMiBpdGVtIDM6IHRoZSBleGFjdCBzdGFjayBvbiBhIFRlc2xhIFQ0LCBjaGVja2VkIGJlZm9yZSBhbnkgYXR0ZW1wdCBvcGVucyBhbmQgaW4gdGhlIEJhc2VsaW5lL0ZpbmFsVGVzdCBub3RlYm9va3M7IGEgbWlzbWF0Y2ggbWVhbnMgbm8gc3R1ZHkgcnVuIiwKICAidHJhaW4iOiB7CiAgICJ0cmFuc2Zvcm1lcnMiOiAiNC41MS4zIiwKICAgImJpdHNhbmRieXRlcyI6ICIwLjUwLjIiLAogICAicGVmdCI6ICIwLjE1LjIiLAogICAiYWNjZWxlcmF0ZSI6ICIxLjYuMCIsCiAgICJodWdnaW5nZmFjZS1odWIiOiAiMC4zMC4yIiwKICAgInRva2VuaXplcnMiOiAiMC4yMS4xIiwKICAgInNhZmV0ZW5zb3JzIjogIjAuNS4zIiwKICAgIm52aWRpYS1tbC1weSI6ICIxMi41NzAuODYiCiAgfSwKICAiZXZhbHVhdGlvbiI6IHsKICAgInRvcmNoIjogIjIuMTEuMCtjdTEzMCIsCiAgICJ0cmFuc2Zvcm1lcnMiOiAiNC41MS4zIiwKICAgImJpdHNhbmRieXRlcyI6ICIwLjUwLjIiLAogICAiYWNjZWxlcmF0ZSI6ICIxLjYuMCIsCiAgICJodWdnaW5nZmFjZS1odWIiOiAiMC4zMC4yIiwKICAgInRva2VuaXplcnMiOiAiMC4yMS4xIiwKICAgInNhZmV0ZW5zb3JzIjogIjAuNS4zIgogIH0sCiAgImFkYXB0ZXJfZXZhbHVhdGlvbiI6IHsKICAgInBlZnQiOiAiMC4xNS4yIgogIH0KIH0KfQpgYGAK').decode('utf-8')
MANIFEST_SHA256 = '414b67eb43db3d7bffd052452634e772e592d1de8e6741e65a73384700db5787'
assert hashlib.sha256(MANIFEST_TEXT.encode('utf-8')).hexdigest() == MANIFEST_SHA256, 'Embedded IMPLEMENTATION_MANIFEST.md was changed: refusing to run'
MANIFEST = json.loads(MANIFEST_TEXT.split('```json\n', 1)[1].split('\n```', 1)[0])
assert MANIFEST['preregistration']['sha256'] == 'b2e6df8f552dfe5756cfde9308880f06de0685d7d1d40de57c5e71cc41260396', 'Manifest is for another PREREGISTRATION.md: refusing to run'
assert MANIFEST['notebook_payloads']['Phase2_Baseline_DEV.ipynb'] == hashlib.sha256(blob).hexdigest(), 'Notebook payload differs from the implementation manifest: refusing to run'
for member, source in {'hinglish_pilot/__init__.py': 'test1_pilot/hinglish_pilot/__init__.py', 'hinglish_pilot/data.py': 'test1_pilot/hinglish_pilot/data.py', 'hinglish_pilot/engine.py': 'test1_pilot/hinglish_pilot/engine.py', 'hinglish_pilot/hf_backend.py': 'test1_pilot/hinglish_pilot/hf_backend.py', 'hinglish_pilot/report.py': 'test1_pilot/hinglish_pilot/report.py', 'hinglish_pilot/scorer_v1.py': 'test1_pilot/hinglish_pilot/scorer_v1.py', 'secondary_analysis/__init__.py': 'secondary_analysis/__init__.py', 'secondary_analysis/cmi.py': 'secondary_analysis/cmi.py', 'secondary_analysis/numeric_endpoint.py': 'secondary_analysis/numeric_endpoint.py', 'secondary_analysis/report_secondary.py': 'secondary_analysis/report_secondary.py', 't4_smoke/train_smoke.py': 't4_smoke/train_smoke.py', 't4_smoke/labels.py': 't4_smoke/labels.py', 't4_smoke/protocol.py': 't4_smoke/protocol.py', 't4_smoke/train_phase2.py': 't4_smoke/train_phase2.py', 't4_smoke/phase2_eval.py': 't4_smoke/phase2_eval.py', 'inputs/dev.v2.json': 'phase2/dev.v2.json'}.items():
    assert hashlib.sha256((PACKAGE / member).read_bytes()).hexdigest() == MANIFEST['files'][source], f'{member} differs from the implementation manifest: refusing to run'
(INPUTS / 'IMPLEMENTATION_MANIFEST.md').write_text(MANIFEST_TEXT, encoding='utf-8')
print('Implementation manifest verified:', MANIFEST_SHA256)
if USE_DRIVE and Path('/content').exists():  # Drive only inside Colab
    from google.colab import drive
    drive.mount('/content/drive')
    STORE = Path('/content/drive/MyDrive/Hinglish_Phase2')
else:
    STORE = PACKAGE / 'runs'
STORE.mkdir(parents=True, exist_ok=True)
try:  # Qwen3-4B is ungated; a token only avoids anonymous rate limits.
    from google.colab import userdata
    os.environ['HF_TOKEN'] = (userdata.get('HF_TOKEN') or '').strip() or os.environ.get('HF_TOKEN', '')
except Exception as error:
    print('No HF_TOKEN Colab secret (' + type(error).__name__ + '); downloading anonymously.')
if not os.environ.get('HF_TOKEN'):
    os.environ.pop('HF_TOKEN', None)

def step(log_dir, script, *args):
    """Run one phase in a fresh Python process, stream its output and log the attempt."""
    log_dir = Path(log_dir)
    log_dir.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, '-u', str(PACKAGE / 't4_smoke' / script), *map(str, args)]
    started, returncode = time.time(), None
    try:
        with open(log_dir / 'log.txt', 'a') as log, subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                                                     text=True, errors='replace') as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
            except BaseException:  # interrupt: never leave a phase running on the GPU behind the notebook
                process.kill()
                process.wait()
                log.write('[notebook] interrupted; phase process killed\n')
                raise
        returncode = process.returncode
    finally:
        with open(log_dir / 'attempts.jsonl', 'a') as attempts:
            attempts.write(json.dumps({'script': script, 'args': list(map(str, args)), 'returncode': returncode,
                                       'started': started, 'seconds': time.time() - started}) + '\n')
    return returncode

def zip_results(folder, archive, prefix):
    """All small files under ``folder``; weights, optimizer state and per-attempt journals stay out (hashes listed)."""
    files, large = [], {}
    for path in sorted(Path(folder).rglob('*')):
        if not path.is_file() or 'journal' in path.relative_to(folder).parts or path.name.endswith('.lock'):
            continue
        if path.suffix in ('.pt', '.safetensors'):
            large[str(path.relative_to(folder))] = hashlib.sha256(path.read_bytes()).hexdigest()
        else:
            files.append(path)
    sums = ''.join(f'{hashlib.sha256(p.read_bytes()).hexdigest()}  {p.relative_to(folder)}\n' for p in files)
    sums += ''.join(f'{digest}  {name}  (not included)\n' for name, digest in large.items())
    with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
        for path in files:
            z.write(path, str(Path(prefix) / path.relative_to(folder)))
        z.writestr(str(Path(prefix) / 'SHA256SUMS.txt'), sums)
    print(sums)
    print('Return:', archive, 'SHA-256', hashlib.sha256(Path(archive).read_bytes()).hexdigest())
    try:
        from google.colab import files as colab_files
        colab_files.download(str(archive))
    except ImportError:
        print('Local: ZIP saved on disk.')


In [ ]:
try:
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
except FileNotFoundError:  # CPU runtime: no nvidia-smi at all
    gpu = ''
print('GPU:', gpu or 'none')
assert 'T4' in gpu, 'Select Runtime → Change runtime type → T4 GPU'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PACKAGE / 'requirements-phase2.txt')], check=True)
# A fresh interpreter sees the installed versions; this kernel never imports torch.
STACK = json.loads(subprocess.run([sys.executable, '-c', 'import sys; sys.path.insert(0, sys.argv[1]); import train_smoke as t, json; '
                                   'v = t.versions(); print(json.dumps({"versions": v, "mismatches": t.stack_mismatches(v)}))',
                                   str(PACKAGE / 't4_smoke')], capture_output=True, text=True, check=True).stdout)
print(json.dumps(STACK['versions'], indent=1))
STACK_MISMATCHES = STACK['mismatches']
print('D021 stack mismatches:', STACK_MISMATCHES or 'none')
# Amendment 2 item 3: only the exact §2 stack on a T4 can produce a study run; there is no override.
assert not STACK_MISMATCHES, 'D021 stack mismatch: no study run until a dated amendment (Amendment 2 item 3): ' + '; '.join(STACK_MISMATCHES)
# Tokenizer, chat template and config files at the pinned revision must match the implementation manifest.
HUB_CHECK = """
import hashlib, json, os, sys
from huggingface_hub import hf_hub_download
model = json.loads(sys.argv[1])
for name, expected in sorted(model['hub_files'].items()):
    path = hf_hub_download(model['id'], name, revision=model['revision'], token=os.environ.get('HF_TOKEN') or None)
    actual = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    assert actual == expected, f'{name} at the pinned revision has SHA-256 {actual}, manifest {expected}: refusing to run'
print('Pinned Hub files match the implementation manifest:', ', '.join(sorted(model['hub_files'])))
"""
print(subprocess.run([sys.executable, '-c', HUB_CHECK, json.dumps(MANIFEST['model'])], capture_output=True, text=True, check=True).stdout)


In [ ]:
RUN_DIR = STORE / 'baseline_dev' / 'base'
step(RUN_DIR.parent, 'phase2_eval.py', 'evaluate', '--split', 'dev', '--inputs', INPUTS, '--run-dir', RUN_DIR,
     '--implementation-manifest', INPUTS / 'IMPLEMENTATION_MANIFEST.md', *(['--recover-stale-lock'] if RECOVER_STALE_LOCK else []))
FLAG = RUN_DIR.parent / 'base_ceiling_flag.json'
RECORD = RUN_DIR.parent / 'base_baseline_record.json'
assert FLAG.exists() and RECORD.exists(), 'Evaluation incomplete (see above): re-run all to resume, then download'
flag = json.loads(FLAG.read_text())
low, high = flag['wilson95']
print(f"\n{flag['name']}: {'FIRES' if flag['flag'] else 'does not fire'} — unchanged DEV C {flag['k']}/{flag['n']}"
      f" (Wilson 95% {100 * low:.1f}–{100 * high:.1f}%; threshold >= 38/40)")
record = json.loads(RECORD.read_text())
assert record['implementation_manifest_sha256'] == MANIFEST_SHA256, 'Baseline was recorded under another implementation manifest'
print('Unchanged-DEV baseline recorded for training (Amendment 2 item 6):', record['results_zip'], record['results_zip_sha256'])


## Download
If the run stopped early, re-run all cells first. Then run this cell.


In [ ]:
zip_results(STORE / 'baseline_dev', PACKAGE / 'Phase2_Baseline_DEV_results.zip', 'baseline_dev')
